<a href="https://colab.research.google.com/github/halimAhtasham/DeepLearning/blob/main/07_Hyperparameter_Tuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Phase 7: Hyperparameter tuning**

### **Objective**

Optimize the baseline machine learning models using cross-validation
and systematic hyperparameter search.

Models:
1. Logistic Regression
2. K-Nearest Neighbors
3. Decision Tree
4. Random Forest

Methods:
- GridSearchCV
- Stratified 5-Fold Cross-Validation

The final test set will remain untouched during hyperparameter tuning.

In [1]:
import pandas as pd
import numpy as np

from google.colab import drive

from sklearn.model_selection import (
    train_test_split,
    StratifiedKFold,
    GridSearchCV
)

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [2]:
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
PROJECT_PATH = "/content/drive/MyDrive/Machine Learning/Projects/Heart Disease Prediction"

DATASET_PATH = PROJECT_PATH + "/Dataset"
RESULT_PATH = PROJECT_PATH + "/Results"
MODEL_PATH = PROJECT_PATH + "/Models"

## **Dataset Load**

In [4]:
df = pd.read_csv(DATASET_PATH + "/heart.csv")

df_clean = df.drop_duplicates().copy()

print("Original dataset:", df.shape)
print("Cleaned dataset:", df_clean.shape)

Original dataset: (1025, 14)
Cleaned dataset: (302, 14)


## **Separate Features**

In [5]:
X= df_clean.drop("target", axis=1)
y=df_clean["target"]

## **Train / Test Split**

In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [7]:
print("Training:", X_train.shape)
print("Test:", X_test.shape)

Training: (241, 13)
Test: (61, 13)


## **Stratified 5-Fold Cross Validation**

In [9]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

# **Logistic Regression Hyperparameter Tuning**

In [10]:
logistic_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(random_state=42, max_iter=2000))
])

In [11]:
logistic_params = {
    "model__C":[0.01,0.1,1,10,100]
}

In [12]:
logistic_grid = GridSearchCV(
    estimator=logistic_pipeline,
    param_grid=logistic_params,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    return_train_score=False
)

In [13]:
logistic_grid.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model',
                                        LogisticRegression(max_iter=2000,
                                                           random_state=42))]),
             n_jobs=-1, param_grid={'model__C': [0.01, 0.1, 1, 10, 100]},
             scoring='f1')

In [14]:
print("Best Parameters:")
print(logistic_grid.best_params_)

Best Parameters:
{'model__C': 1}


In [15]:
print("Best CV F1:")
print(logistic_grid.best_score_)

Best CV F1:
0.8600187148574244


In [16]:
best_logistic = logistic_grid.best_estimator_

In [17]:
logistic_results = pd.DataFrame(
    logistic_grid.cv_results_
)

logistic_results[
    [
        "param_model__C",
        "mean_test_score",
        "std_test_score",
        "rank_test_score"
    ]
].sort_values(
    "rank_test_score"
)

,param_model__C,mean_test_score,std_test_score,rank_test_score
2,1.00,0.860019,0.068267,1
3,10.00,0.854953,0.062324,2
4,100.00,0.854953,0.062324,2
1,0.10,0.854147,0.063010,4
0,0.01,0.848444,0.054938,5


# **KNN Hyperparameter Tuning**

In [18]:
knn_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", KNeighborsClassifier())
])

In [19]:
knn_params = {
    "model__n_neighbors": [3, 5, 7, 9, 11, 15, 21],
    "model__weights": ["uniform", "distance"],
    "model__p": [1, 2]
}

In [20]:
knn_grid = GridSearchCV(
    estimator=knn_pipeline,
    param_grid=knn_params,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    return_train_score=False
)

knn_grid.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=Pipeline(steps=[('scaler', StandardScaler()),
                                       ('model', KNeighborsClassifier())]),
             n_jobs=-1,
             param_grid={'model__n_neighbors': [3, 5, 7, 9, 11, 15, 21],
                         'model__p': [1, 2],
                         'model__weights': ['uniform', 'distance']},
             scoring='f1')

In [21]:
print("Best Parameters:")
print(knn_grid.best_params_)

print("\nBest CV F1:")
print(knn_grid.best_score_)

Best Parameters:
{'model__n_neighbors': 21, 'model__p': 1, 'model__weights': 'uniform'}

Best CV F1:
0.8753377270810008


In [22]:
best_knn = knn_grid.best_estimator_

# **Decision Tree Hyperparameter Tuning**

In [23]:
tree_params = {
    "max_depth": [2, 3, 4, 5, 6, 8, 10, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4, 6]
}

In [24]:
tree_model = DecisionTreeClassifier(
    random_state=42
)

In [25]:
tree_grid = GridSearchCV(
    estimator=tree_model,
    param_grid=tree_params,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    return_train_score=False
)

tree_grid.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=DecisionTreeClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [2, 3, 4, 5, 6, 8, 10, None],
                         'min_samples_leaf': [1, 2, 4, 6],
                         'min_samples_split': [2, 5, 10]},
             scoring='f1')

In [26]:
print("Best Parameters:")
print(tree_grid.best_params_)

print("\nBest CV F1:")
print(tree_grid.best_score_)

Best Parameters:
{'max_depth': 4, 'min_samples_leaf': 6, 'min_samples_split': 2}

Best CV F1:
0.8216529233478387


In [27]:
best_tree = tree_grid.best_estimator_

# **Random Forest Hyperparameter Tuning**

In [28]:
forest_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 3, 5, 7, 10],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

In [29]:
forest_model = RandomForestClassifier(
    random_state=42
)

In [30]:
forest_grid = GridSearchCV(
    estimator=forest_model,
    param_grid=forest_params,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    return_train_score=False
)

In [31]:
forest_grid.fit(X_train, y_train)

GridSearchCV(cv=StratifiedKFold(n_splits=5, random_state=42, shuffle=True),
             estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [None, 3, 5, 7, 10],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='f1')

In [32]:
print("Best Parameters:")
print(forest_grid.best_params_)

print("\nBest CV F1:")
print(forest_grid.best_score_)

Best Parameters:
{'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}

Best CV F1:
0.875200743062812


In [33]:
best_forest = forest_grid.best_estimator_

In [34]:
tuning_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "KNN",
        "Decision Tree",
        "Random Forest"
    ],

    "Best CV F1": [
        logistic_grid.best_score_,
        knn_grid.best_score_,
        tree_grid.best_score_,
        forest_grid.best_score_
    ]
})

tuning_results = tuning_results.sort_values(
    "Best CV F1",
    ascending=False
).reset_index(drop=True)

tuning_results

,Model,Best CV F1
0,KNN,0.875338
1,Random Forest,0.875201
2,Logistic Regression,0.860019
3,Decision Tree,0.821653


In [35]:
tuning_results["Best CV F1"] = tuning_results["Best CV F1"].round(4)

tuning_results

,Model,Best CV F1
0,KNN,0.8753
1,Random Forest,0.8752
2,Logistic Regression,0.8600
3,Decision Tree,0.8217


In [36]:
print("Logistic Regression")
print(logistic_grid.best_params_)

print("\nKNN")
print(knn_grid.best_params_)

print("\nDecision Tree")
print(tree_grid.best_params_)

print("\nRandom Forest")
print(forest_grid.best_params_)

Logistic Regression
{'model__C': 1}

KNN
{'model__n_neighbors': 21, 'model__p': 1, 'model__weights': 'uniform'}

Decision Tree
{'max_depth': 4, 'min_samples_leaf': 6, 'min_samples_split': 2}

Random Forest
{'max_depth': 5, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}


In [37]:
baseline_f1 = {
    "Logistic Regression": 0.8600187148574244,
    "KNN": 0.8474228634850165,
    "Decision Tree": 0.804037786774629,
    "Random Forest": 0.8666440371941861
}

In [38]:
comparison = tuning_results.copy()

comparison["Baseline CV F1"] = comparison["Model"].map(
    baseline_f1
)

comparison["Improvement"] = (
    comparison["Best CV F1"]
    - comparison["Baseline CV F1"]
)

comparison

,Model,Best CV F1,Baseline CV F1,Improvement
0,KNN,0.8753,0.847423,0.027877
1,Random Forest,0.8752,0.866644,0.008556
2,Logistic Regression,0.8600,0.860019,-0.000019
3,Decision Tree,0.8217,0.804038,0.017662


In [39]:
tuning_results.to_csv(
    RESULT_PATH + "/hyperparameter_tuning_results.csv",
    index=False
)

comparison.to_csv(
    RESULT_PATH + "/baseline_vs_tuned.csv",
    index=False
)

In [40]:
best_parameters = {
    "Logistic Regression": logistic_grid.best_params_,
    "KNN": knn_grid.best_params_,
    "Decision Tree": tree_grid.best_params_,
    "Random Forest": forest_grid.best_params_
}

best_parameters

{'Logistic Regression': {'model__C': 1},
 'KNN': {'model__n_neighbors': 21, 'model__p': 1, 'model__weights': 'uniform'},
 'Decision Tree': {'max_depth': 4,
  'min_samples_leaf': 6,
  'min_samples_split': 2},
 'Random Forest': {'max_depth': 5,
  'max_features': 'sqrt',
  'min_samples_leaf': 1,
  'min_samples_split': 2,
  'n_estimators': 100}}

In [41]:
import json

with open(
    RESULT_PATH + "/best_parameters.json",
    "w"
) as f:
    json.dump(best_parameters, f, indent=4)

In [42]:
import joblib

joblib.dump(
    best_logistic,
    MODEL_PATH + "/tuned_logistic_regression.pkl"
)

joblib.dump(
    best_knn,
    MODEL_PATH + "/tuned_knn.pkl"
)

joblib.dump(
    best_tree,
    MODEL_PATH + "/tuned_decision_tree.pkl"
)

joblib.dump(
    best_forest,
    MODEL_PATH + "/tuned_random_forest.pkl"
)

['/content/drive/MyDrive/Machine Learning/Projects/Heart Disease Prediction/Models/tuned_random_forest.pkl']

In [44]:
import os
os.listdir(MODEL_PATH)

['scaler.pkl',
 'logistic_regression.pkl',
 'tuned_logistic_regression.pkl',
 'tuned_knn.pkl',
 'tuned_decision_tree.pkl',
 'tuned_random_forest.pkl']

# Hyperparameter Tuning Findings

Hyperparameter tuning was performed using GridSearchCV with Stratified
5-Fold Cross-Validation. F1-score was selected as the primary optimization
metric.

The tuned KNN model achieved the highest cross-validated F1-score of
0.8753, followed very closely by Random Forest with 0.8752.

Compared with the baseline models, KNN showed the largest improvement,
while Logistic Regression remained unchanged with C=1.

The tuned Decision Tree used a maximum depth of 4 and a minimum leaf size
of 6, indicating that limiting model complexity improved its performance.

Although KNN achieved the highest F1-score, its advantage over Random
Forest was only 0.0001. Therefore, KNN and Random Forest will be treated
as the primary candidates for final model selection.

The final test set has not been used during hyperparameter tuning and
remains reserved for the final evaluation.